In [1]:
from pathlib import Path
from datetime import datetime
import re

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from Define_Dataset_SM_Global import MRIDataset_SM
from SFCN_DGM import SFCN

ROOT = Path(r"C:\Lab\test_retest_dataset")
OUTPUT_DIR = Path(r"C:\Lab\models\SFCN\sm\output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = Path(r"C:\Lab\models\SFCN\sm\pretrained\pretrain_regionalFOV_SM_DGM_withcc_v2_20260305.pt")
DATASETS = ["Maclaren", "MMRR", "OAS1"]
BATCH_SIZE = 4
OUTPUT_DIM = 95

CROP_MIN = np.array([52, 77, 50])
CROP_MAX = np.array([140, 168, 122])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
age_centers = torch.tensor(np.arange(5.5, 100, 1), dtype=torch.float32, device=device)
print(f"Device: {device}")

c:\Users\user\miniconda3\envs\lib\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
model = SFCN(output_dim=OUTPUT_DIM).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True), strict=True)
model.eval()

for dataset_name in DATASETS:
    image_dir = ROOT / dataset_name / "ndf" / "dgm16"
    label_file = ROOT / dataset_name / f"{dataset_name}.xlsx"
    labels = pd.read_excel(label_file)
    if not {"Subject_ID", "Age"}.issubset(labels.columns):
        raise ValueError(f"{label_file}: 缺少 Subject_ID 或 Age 欄位")

    labels = labels[["Subject_ID", "Age"]].copy()
    labels["Subject_ID"] = labels["Subject_ID"].astype(str).str.strip()
    labels["Filename"] = labels["Subject_ID"] + "_dgm16.nii"
    missing = [name for name in labels["Filename"] if not (image_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"{dataset_name} 缺少 {len(missing)} 張影像，例如：{missing[:3]}")

    dataset = MRIDataset_SM(
        data_folders_set=[str(image_dir)],
        labels_df=labels[["Filename", "Age"]].reset_index(drop=True),
        bin_range=(5, 100),
        crop_coords=(CROP_MIN, CROP_MAX),
    )
    if len(dataset) != len(labels):
        raise ValueError(f"{dataset_name}: Dataset 數量與 Excel 不一致")

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=(device.type == "cuda"))
    true_ages, pred_ages = [], []

    with torch.no_grad():
        for images, ages, _ in tqdm(loader, desc=dataset_name):
            images = images.to(device, dtype=torch.float32)
            output = model(images)[0].reshape(images.shape[0], -1)
            if output.shape[1] != OUTPUT_DIM:
                raise ValueError(f"模型輸出維度不符：{tuple(output.shape)}")
            predicted = (output.exp() * age_centers).sum(dim=1)
            true_ages.extend(ages.cpu().numpy().tolist())
            pred_ages.extend(predicted.cpu().numpy().tolist())

    if len(pred_ages) != len(labels):
        raise ValueError(f"{dataset_name}: 預測數量不符")

    result = pd.DataFrame({
        "Subject ID": labels["Subject_ID"].to_numpy(),
        "True Age": true_ages,
        "Predicted Age": pred_ages,
    })
    mae = np.mean(np.abs(result["True Age"] - result["Predicted Age"]))
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output = OUTPUT_DIR / f"SFCN_sm_{dataset_name}_dgm_{BATCH_SIZE}_{timestamp}.xlsx"
    result.to_excel(output, index=False)
    print(f"{dataset_name}: N={len(result)}, MAE={mae:.4f}, Saved: {output}")

Maclaren: 100%|██████████| 30/30 [00:07<00:00,  4.28it/s]


Maclaren: N=120, MAE=3.3037, Saved: C:\Lab\models\SFCN\sm\output\SFCN_sm_Maclaren_dgm_4_20261107_203556.xlsx


MMRR: 100%|██████████| 11/11 [00:02<00:00,  4.78it/s]


MMRR: N=42, MAE=6.1862, Saved: C:\Lab\models\SFCN\sm\output\SFCN_sm_MMRR_dgm_4_20261107_203558.xlsx


OAS1: 100%|██████████| 10/10 [00:01<00:00,  5.77it/s]

OAS1: N=40, MAE=3.8082, Saved: C:\Lab\models\SFCN\sm\output\SFCN_sm_OAS1_dgm_4_20261107_203600.xlsx


In [3]:
# 合併同一 batch size 的最新結果；每個資料集一個工作表。
pattern = re.compile(
    rf"^SFCN_sm_(Maclaren|MMRR|OAS1)_dgm_{BATCH_SIZE}_(\d{{8}}_\d{{6}})\.xlsx$"
)
latest = {}
for path in OUTPUT_DIR.glob("SFCN_sm_*_dgm_*.xlsx"):
    match = pattern.fullmatch(path.name)
    if match:
        dataset_name, timestamp = match.groups()
        if dataset_name not in latest or timestamp > latest[dataset_name][0]:
            latest[dataset_name] = (timestamp, path)

if not latest:
    raise FileNotFoundError(f"找不到 batch size={BATCH_SIZE} 的 DGM 預測 Excel")

merged_path = OUTPUT_DIR / f"SFCN_sm_dgm_merged_{BATCH_SIZE}.xlsx"
with pd.ExcelWriter(merged_path, engine="openpyxl") as writer:
    for dataset_name in DATASETS:
        if dataset_name not in latest:
            print(f"未找到：{dataset_name}")
            continue
        file_path = latest[dataset_name][1]
        df = pd.read_excel(file_path)
        expected = ["Subject ID", "True Age", "Predicted Age"]
        if not set(expected).issubset(df.columns):
            raise ValueError(f"{file_path.name}: 缺少必要欄位")
        df = df[expected].copy()
        df.insert(0, "Dataset", dataset_name)
        df.to_excel(writer, sheet_name=f"{dataset_name}_dgm", index=False)
        print(f"{dataset_name}_dgm <- {file_path.name}")

print(f"合併完成：{merged_path}")

Maclaren_dgm <- SFCN_sm_Maclaren_dgm_4_20261107_203556.xlsx
MMRR_dgm <- SFCN_sm_MMRR_dgm_4_20261107_203558.xlsx
OAS1_dgm <- SFCN_sm_OAS1_dgm_4_20261107_203600.xlsx
合併完成：C:\Lab\models\SFCN\sm\output\SFCN_sm_dgm_merged_4.xlsx
